# Tool-calling fine-tuning smoke test

This notebook runs the repository's complete tool-calling smoke pipeline on a remote GPU: dataset generation, baseline, two QLoRA optimizer steps, adapter reload, and execution-aware evaluation.

Before running the cells, select **Runtime → Change runtime type → T4 GPU**. GPU availability depends on Colab capacity.

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
!test -d llm-finetuning || git clone https://github.com/lepeutj/llm-finetuning.git
%cd /content/llm-finetuning
!git pull --ff-only

In [ ]:
!python3 -m pip install -q uv
!uv python install 3.12
!PYTHON_BIN="$(uv python find 3.12)" bash setup.sh --backend cuda

In [ ]:
!MODEL_PROFILE=qwen-0.5b QUANTIZATION=nf4 bash run_tool_experiment.sh smoke

In [ ]:
import json
from pathlib import Path
files = sorted(Path('results/tool_calling').glob('*/comparison.json'))
assert files, 'No comparison file was produced.'
result_file = files[-1]
print(result_file)
print(json.dumps(json.loads(result_file.read_text()), indent=2, ensure_ascii=False))

## Next run

The smoke run is deliberately partial. For the complete 300-example benchmark, use:

```bash
MODEL_PROFILE=qwen-0.5b QUANTIZATION=nf4 bash run_tool_experiment.sh full
```

After training, `python -m src.tool_agent` runs a complete route, tool execution, and final-response loop.